In [3]:
!pip show tensorflow

Name: tensorflow
Version: 2.9.3
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: c:\users\alenj\appdata\local\programs\python\python310\lib\site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, keras-preprocessing, libclang, numpy, opt-einsum, packaging, protobuf, setuptools, six, tensorboard, tensorflow-estimator, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: 


In [5]:
!pip uninstall tensorflow -y
!pip uninstall keras -y

Found existing installation: tensorflow 2.9.3
Uninstalling tensorflow-2.9.3:
  Successfully uninstalled tensorflow-2.9.3


You can safely remove it manually.


Found existing installation: keras 2.9.0
Uninstalling keras-2.9.0:
  Successfully uninstalled keras-2.9.0


In [6]:
!pip install tensorflow==2.15.0


  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
  Using cached protobuf-4.25.8-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/300.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/300.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/300.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/300.9 MB 1.3 MB/s eta 0:03:53
   ---------------------------------------- 0.8/300.9 MB 1.3 MB/s eta 0:03:53
   ---------------------------------------- 1.0/300.9 MB 1.2 MB/s eta 0:04:04
   ---------------------------------------- 1.3/300.9 MB 1.2 MB/s eta 0:04:06
   ---------------------------------------- 1.6/300.9 MB 1.2 MB/s eta 0:04:03
   ---------------------------------------- 1.8/300.9 MB 1.2 MB/s eta 0:04:04
   -----------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
# TRAIN_MOBILENETV2_3CLASS.py
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os

# Paths
DATA_DIR = "dataset"  # <- change if needed
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparams
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
NUM_CLASSES = 3  # normal, benign, malignant

# Data generators (80% train / 20% val)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
)

val_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
)

print("Class indices:", train_gen.class_indices)

# Model: MobileNetV2 base
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False  # freeze base initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
preds = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=preds)

model.compile(optimizer=Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Callbacks
checkpoint_path = os.path.join(OUTPUT_DIR, "mobilenetv2_best.h5")
callbacks = [
    ModelCheckpoint(
        checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

# Train (transfer learning)
history = model.fit(
    train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks, verbose=1
)

# Optional: fine-tune last few layers
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"]
)
ft_history = model.fit(
    train_gen, validation_data=val_gen, epochs=5, callbacks=callbacks, verbose=1
)

# Save final Keras model
keras_path = os.path.join(OUTPUT_DIR, "mobilenetv2_3class.h5")
model.save(keras_path)
print("Saved Keras model:", keras_path)

# -------- Convert to TFLite (float32) --------
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = os.path.join(OUTPUT_DIR, "mobilenetv2_3class.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("Saved TFLite (float32):", tflite_path, "size:", os.path.getsize(tflite_path))


# -------- Convert to INT8 Quantized (optional) --------
# Build a small representative dataset generator for quantization
def representative_data_gen():
    for i in range(100):
        img_batch, _ = next(train_gen)  # already rescaled to [0,1]
        # Use only the first image in batch
        yield [img_batch[0:1].astype("float32")]


converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
converter_q.representative_dataset = representative_data_gen
converter_q.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_q.inference_input_type = tf.uint8  # or tf.int8 depending on flow
converter_q.inference_output_type = tf.uint8
try:
    tflite_quant = converter_q.convert()
    tflite_quant_path = os.path.join(OUTPUT_DIR, "mobilenetv2_3class_int8.tflite")
    with open(tflite_quant_path, "wb") as f:
        f.write(tflite_quant)
    print(
        "Saved INT8 quantized TFLite:",
        tflite_quant_path,
        "size:",
        os.path.getsize(tflite_quant_path),
    )
except Exception as e:
    print("Quant conversion failed:", e)

# -------- Quick validation with TFLite interpreter --------
print("Validating TFLite (float32) model...")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:", input_details)
print("Output details:", output_details)

# Run a single sample from validation generator
sample_img, sample_label = next(val_gen)
sample = sample_img[0:1].astype("float32")  # (1,224,224,3)
interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()
out = interpreter.get_tensor(output_details[0]["index"])
print("TFLite output:", out)


Found 1263 images belonging to 3 classes.
Found 315 images belonging to 3 classes.
Class indices: {'benign': 0, 'malignant': 1, 'normal': 2}


9406464/9406464 [==============================] - 8s 1us/step
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                 

c:\Users\alenj\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


79/79 [==============================] - 59s 712ms/step - loss: 0.7882 - accuracy: 0.6722 - val_loss: 0.5572 - val_accuracy: 0.7778 - lr: 1.0000e-04
Epoch 2/15
79/79 [==============================] - ETA: 0s - loss: 0.5765 - accuracy: 0.7498
Epoch 2: val_accuracy did not improve from 0.77778
79/79 [==============================] - 24s 297ms/step - loss: 0.5765 - accuracy: 0.7498 - val_loss: 0.5590 - val_accuracy: 0.7587 - lr: 1.0000e-04
Epoch 3/15
79/79 [==============================] - ETA: 0s - loss: 0.5016 - accuracy: 0.7688
Epoch 3: val_accuracy did not improve from 0.77778
79/79 [==============================] - 25s 315ms/step - loss: 0.5016 - accuracy: 0.7688 - val_loss: 0.4848 - val_accuracy: 0.7619 - lr: 1.0000e-04
Epoch 4/15
79/79 [==============================] - ETA: 0s - loss: 0.4478 - accuracy: 0.8108
Epoch 4: val_accuracy did not improve from 0.77778
79/79 [==============================] - 25s 313ms/step - loss: 0.4478 - accuracy: 0.8108 - val_loss: 0.5411 - val_acc

INFO:tensorflow:Assets written to: C:\Users\alenj\AppData\Local\Temp\tmpq5fkgkug\assets


Saved TFLite (float32): results\mobilenetv2_3class.tflite size: 9515532
INFO:tensorflow:Assets written to: C:\Users\alenj\AppData\Local\Temp\tmp351bcq31\assets


INFO:tensorflow:Assets written to: C:\Users\alenj\AppData\Local\Temp\tmp351bcq31\assets
c:\Users\alenj\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved INT8 quantized TFLite: results\mobilenetv2_3class_int8.tflite size: 2868352
Validating TFLite (float32) model...
Input details: [{'name': 'serving_default_input_1:0', 'index': 0, 'shape': array([  1, 224, 224,   3]), 'shape_signature': array([ -1, 224, 224,   3]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall:0', 'index': 180, 'shape': array([1, 3]), 'shape_signature': array([-1,  3]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
TFLite output: [[0.8849935  0.02452421 0.09048234]]
